# Normal Chain

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.0
)

parser = StrOutputParser()

template = PromptTemplate.from_template("You are a helpful assistant. Answer the question: {question}")
chain = template | llm | parser
result = chain.invoke({"question": "What is the capital of France?"})

print(result)

The capital of France is Paris.


In [ ]:
from json import load
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/deepseek-v4-pro",
    temperature=0.0,
    api_key=os.getenv("DEEPSEEK_API_KEY_PRO"),
    base_url="https://integrate.api.nvidia.com/v1"
)

parser = StrOutputParser()
template = PromptTemplate.from_template("You are a helpful assistant. Answer the question: {question}")
chain = template | llm | parser

result = chain.invoke({"question": "What is the capital of France?"})
print(result)

In [4]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


# Sequential Chain

In [5]:
from json import load
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()
llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.0
)

parser = StrOutputParser()

seq_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

seq_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key pints on it \n {summary}""")

seq_chain = seq_template1 | llm | parser | seq_template2 | llm | parser

seq_result = seq_chain.invoke({"topic": "Generative AI"})

print(seq_result)

Based on the summary provided, here are 5 key points about Generative AI:

1. **Focus on Creation, Not Classification:** Generative AI's primary function is to create new and original content (like writing or drawing), rather than simply analyzing, sorting, or classifying existing data.
2. **Learning from Massive Data:** These models learn complex patterns by training on enormous datasets (text, images, code), allowing them to generate novel and contextually relevant outputs.
3. **Diverse Output Capabilities:** Generative AI can produce various forms of content, including text (articles, code), images (art, graphics), audio (music), and programming assistance.
4. **Powered by Core Technologies:** Key examples of this technology include Large Language Models (LLMs) like GPT-4 and diffusion models, which are the core engines behind content generation.
5. **Transformative Impact:** Generative AI is rapidly changing various sectors by automating content creation, making complex creative ta

In [6]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

# Parallel Chain

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv

load_dotenv()

llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.0
)

parser = StrOutputParser()

par_template1 = PromptTemplate.from_template("""
Give me a short summary on topic : {topic}""")

par_template2 = PromptTemplate.from_template("""
Based on the summary, give me 5 key pints on it \n {topic}""")

par_chain1 = par_template1 | llm | parser
par_chain2 = par_template2 | llm | parser


parallel_chain = RunnableParallel(

  {
    "summary": par_chain1,
    "key_points": par_chain2
  }
)

mer_template = PromptTemplate.from_template("""
  Combine the following information into a comprehensive output:
                                             
  Summary: {summary}
  Key Points: {key_points}

""")

mer_chain = mer_template | llm | parser

final_chain = parallel_chain | mer_chain

par_result = final_chain.invoke({"topic": "Generative AI"})
print(par_result)

## Generative AI: A Comprehensive Overview

Generative AI represents a revolutionary type of artificial intelligence defined by its ability to **create new and original content** rather than simply analyzing or classifying existing data. These models learn complex patterns from massive datasets to generate novel outputs that are often highly realistic and contextually relevant.

---

### 1. Core Concepts and Mechanism

**Learning from Data:** Generative AI models are trained on enormous volumes of data (text, images, code, audio). This training allows them to understand the underlying structure, style, and relationships within that data, enabling them to predict and generate coherent outputs.

**Creation, Not Classification:** Unlike traditional AI, which sorts or classifies data (e.g., identifying a category), Generative AI actively produces new artifacts (e.g., generating a unique story or an image).

**Core Technology:** Key examples of this technology include Large Language Models 

In [8]:
final_chain.get_graph().print_ascii()

        +-----------------------------------+        
        | Parallel<summary,key_points>Input |        
        +-----------------------------------+        
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOllama |                    | ChatOllama |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

# Conditional Chain

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

class ReviewSentiment(BaseModel):
  sentiment: Literal["positve", "negative", "neutral"] = Field(description=" The sentiment of the movie review")

llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.0
)

str_parser = StrOutputParser()
pydantic_parser = PydanticOutputParser(pydantic_object=ReviewSentiment)

con_template1 = PromptTemplate.from_template("""
    Give me sentiment of the movie review {review} \n
    format_instruction: {format_instruction}
""",
partial_variables={"format_instruction": pydantic_parser.get_format_instructions()}
)

review_chain = con_template1 | llm | pydantic_parser

# print(review_chain.invoke({"review": "This movie was fantastic! The acting was superb and the plot was engaging."}))


pos_template = PromptTemplate.from_template("""
  The review is postive so write a short appreciation: \n {review}
""")

neg_template = PromptTemplate.from_template("""
  The review is negative so write a short critique: \n {review}
""")

neu_template = PromptTemplate.from_template("""
  The review is neutra so write a short balanced remarks: \n {review}
""")

defult_template = PromptTemplate.from_template("""
  I counln not detect the clear sentimen. Provide a netral response: \n {review}
""")


branch_chain = RunnableBranch(
  (lambda x: x.sentiment == "positive", pos_template | llm | str_parser),
  (lambda x: x.sentiment == "negative", neg_template | llm | str_parser),
  (lambda x: x.sentiment == "neutral", neu_template | llm | str_parser),
  defult_template | llm | str_parser
  )

final_con_chain = review_chain | branch_chain

review = "t was okay—some good moments, but nothing particularly memorable. Just an average watch."

print(review_chain.invoke({"review": review}))

final_con_result = final_con_chain.invoke({"review": review})

print(final_con_result)

sentiment='neutral'
Here are a few options for short, balanced remarks, depending on the context of the review:

**General & Objective:**

* "An average experience."
* "It was satisfactory."
* "Mixed feelings overall."
* "Meets expectations."

**Slightly More Detailed:**

* "The product/service was fine, with some room for improvement."
* "A balanced performance; there are both pros and cons."
* "It delivered what was expected, though not exceptionally."
